# Scraping Websites, Topic Modeling, XML Edition

## Python Web Scraping — Core Imports

| Library | Type | Role |
|---|---|---|
| `requests` | 3rd party | Fetches raw HTML from the web |
| `BeautifulSoup` (bs4) | 3rd party | Parses HTML so you can query it |
| `urljoin` (urllib.parse) | built-in | Resolves relative URLs to absolute |
| `time` | built-in | `time.sleep()` adds polite delay between requests |

`requests` gets the HTML; `BeautifulSoup` reads it. Neither knows how to do the other's job.

In [3]:
# the requests library is used to fetch the HTML content of the webpage
import requests
# BeautifulSoup is used to parse the HTML and extract the links
from bs4 import BeautifulSoup
from urllib.parse import urljoin
# built in library: courtesy delay between HTTP requests 
# so you don't hammer the server with dozens of rapid-fire requests
import time

In [4]:
# declare the base URL as a variable
BASE = "https://www.history.pitt.edu"

In [5]:
# Step 1: collect article URLs
links = []
for page in range(2):
    soup = BeautifulSoup(requests.get(f"{BASE}/news?page={page}").content, "lxml")
    links += [urljoin(BASE, a["href"]) for a in soup.select("h3 a") if a["href"].startswith("/")]

# Step 2: scrape each article into a list of dicts
articles = []
for url in links[:30]:
    soup = BeautifulSoup(requests.get(url).content, "lxml")
    articles.append({
        "title": soup.find("h1").get_text(strip=True),
        "body":  " ".join(p.get_text(strip=True) for p in soup.find_all("p")),
        "url":   url
    })
    time.sleep(1)

## Frequency-Based Keyword Extraction

Now let's get a sense of what the most frequently occurring words are: this might be helpful for making an XML edition of the text as a whole.

```python
from collections import Counter
import re

all_text = " ".join(a["body"] for a in articles).lower()
words = re.findall(r'\b[a-z]{4,}\b', all_text)

stopwords = {"that","this","with","from","have","been","their",
             "they","will","also","which","were","about","pittsburgh",
             "university","history","department"}

counts = Counter(w for w in words if w not in stopwords)
counts.most_common(20)
```

- `Counter` is a built-in Python type that counts occurrences of items in a list
- `re.findall(r'\b[a-z]{4,}\b', all_text)` uses regex to extract only words of 4+ characters, which cuts most noise words without needing a full stopword list
- `stopwords` is a manual set of domain-specific words to exclude — add to it as needed
- `counts.most_common(20)` returns the top 20 words as a list of `(word, count)` tuples

In [8]:
from collections import Counter
import re

all_text = " ".join(a["body"] for a in articles).lower()
words = re.findall(r'\b[a-z]{4,}\b', all_text)   # words 4+ chars

stopwords = {"that","this","with","from","have","been","their",
             "they","will","also","which","were","about","pittsburgh",
             "university","history","department"}

counts = Counter(w for w in words if w not in stopwords)
counts.most_common(20)

[('school', 61),
 ('dietrich', 56),
 ('arts', 55),
 ('pittsburghkenneth', 50),
 ('sciencesdepartment', 50),
 ('wesley', 50),
 ('posvar', 50),
 ('hallpittsburgh', 50),
 ('american', 34),
 ('pitt', 30),
 ('award', 28),
 ('visitapply', 25),
 ('ipsum', 25),
 ('goode', 24),
 ('professor', 22),
 ('latin', 22),
 ('black', 22),
 ('studies', 19),
 ('work', 19),
 ('benjamin', 16)]

Now we are going to use another library to convert from Python structure (list of dictionaries) to actual XML. The Saxon processor reads, queries, and transforms XML *that already exists*, so we need to first get it into that format.

`lxml` is Python's standard library for building and manipulating XML. Unlike BeautifulSoup (which is designed for reading and querying), `lxml` can construct a well-formed XML tree from scratch and write it to disk.

```python
from lxml import etree

root = etree.Element("articles")
for a in articles:
    node = etree.SubElement(root, "article", url=a["url"])
    etree.SubElement(node, "title").text = a["title"]
    etree.SubElement(node, "body").text  = a["body"]

etree.ElementTree(root).write("articles.xml", pretty_print=True, 
                               xml_declaration=True, encoding="UTF-8")
```

- `etree.Element("articles")` creates the root element
- `etree.SubElement(parent, tag, attribute=value)` creates a child element — each `<article>` gets the story URL as an attribute
- `.text` assigns the text content of an element
- `.write()` serializes the tree to a file; `pretty_print=True` adds indentation, `xml_declaration=True` adds `<?xml version="1.0"?>` at the top

To preview the output in the notebook without writing to disk:

```python
print(etree.tostring(root, pretty_print=True).decode())
```

In [9]:
from lxml import etree

root = etree.Element("articles")
for a in articles:
    node = etree.SubElement(root, "article", url=a["url"])
    etree.SubElement(node, "title").text = a["title"]
    etree.SubElement(node, "body").text  = a["body"]

etree.ElementTree(root).write("articles.xml", pretty_print=True, 
                               xml_declaration=True, encoding="UTF-8")

## Pitt History vs. Pitt News Scraper — Key Differences

| | Pitt History | Pitt News |
|---|---|---|
| **Index link selector** | `soup.select("h3 a")` | `soup.find_all("a", class_="homeheadline")` |
| **Pagination** | `?page=0`, `?page=1` | Single page (13 results) |
| **Headers required** | No | Yes — site blocks requests without a browser User-Agent |
| **Body selector** | `soup.find_all("p")` | `soup.find("div", id="sno-story-body-content")` |
| **Extra fields** | title, body, url | + author, date |

The main structural difference: the History site wraps article text in plain `<p>` tags, while the Pitt News site wraps everything in a named `<div>`, which actually makes targeting cleaner and avoids picking up footer/nav paragraph noise.

In [17]:
BASE    = "https://pittnews.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

# Step 1: collect article URLs from the culture index page
soup  = BeautifulSoup(requests.get(f"{BASE}/culture/", headers=HEADERS).content, "lxml")
links = [urljoin(BASE, a["href"]) for a in soup.find_all("a", class_="homeheadline")]

print(f"Found {len(links)} links")
print(links[:3])

# Step 2: scrape each article
articles = []
for url in links[:30]:
    soup   = BeautifulSoup(requests.get(url, headers=HEADERS).content, "lxml")
    title  = soup.find("h1").get_text(strip=True) if soup.find("h1") else ""
    body   = soup.find("div", id="sno-story-body-content")
    author = soup.find("span", class_="byline-name")
    date   = soup.find("span", class_="time-wrapper")
    articles.append({
        "title":  title,
        "author": author.get_text(strip=True) if author else "",
        "date":   date.get_text(strip=True)   if date   else "",
        "body":   body.get_text(separator=" ", strip=True) if body else "",
        "url":    url
    })
    time.sleep(1)

Found 13 links
['https://pittnews.com/article/201934/arts-and-entertainment/how-students-are-taking-care-of-themselves-during-finals/', 'https://pittnews.com/article/201929/top-stories/its-like-a-little-oasis-be-a-good-neighbor-celebration-transforms-wpu-into-a-carnival/', 'https://pittnews.com/article/201873/top-stories/live-laugh-lesbian-new-sapphic-cafe-the-soft-spot-opens-in-garfield/']


In [21]:
url  = links[0]
soup = BeautifulSoup(requests.get(url, headers=HEADERS).content, "lxml")
body = soup.find("div", id="sno-story-body-content")
print(body.get_text(separator=" ", strip=True))

Despite excitement over warmer temperatures and upcoming summer break plans, the end of the semester is often a less-than-enjoyable time for college students. Finals week typically brings an influx of stress and even less time for necessities like sleep, proper nutrition, time with loved ones and self-care. In 2023, an Inside Higher Ed survey found that over 50% of students have experienced chronic stress, consistently feeling overwhelmed and pressured. Finals — whether an exam, essay or project — make this worse. Maintaining mental and physical wellness during finals week is difficult, but important. Sean Browne, a junior civil engineering major, said he’s definitely keeping self-care a priority. “I feel like I definitely have time [for taking care of myself],” Browne said. “I’m someone who doesn’t go crazy studying. I mean, I do study, but I also still make sure that I’m eating and getting enough sleep.” It wasn’t always that easy. Browne said he learned this lesson freshman year, wh

None
13
